# 01 - Post-fix re-run on CUDA
Protocol: `results/postfix_rerun_2026-11/PROTOCOL.md` (written before the run). Sections are in
priority order - stop when the A100 time runs out; re-running the notebook from the top skips
every run that already finished (one JSON per run on Drive). Planning estimates per section are
in the cell titles; they come from the October A100 runs and are not results.

The code comes from branch `checkpoint` of `github.com/dimitrijepesic/Pasado`, checked out at the
pinned commit (alternatively a `checkpoint.bundle` on Drive, see `colab/README.md`).
Runtime: GPU, A100, high RAM.

In [ ]:
# 0.1 Setup: Google Drive, the code at the pinned commit, pinned packages (~3-5 min)
import json, os, subprocess, sys
COMMIT = "91ebd88a06d6eee73a1656dba7f122c84a7664a4"          # code under test (branch checkpoint)
BRANCH = "checkpoint"
REPO_URL = "https://github.com/dimitrijepesic/Pasado.git"   # default source of the code
DRIVE_ROOT = "/content/drive/MyDrive/pasado_checkpoint_2026_11"
BUNDLE = DRIVE_ROOT + "/checkpoint.bundle"   # optional: used instead of GitHub if present
LOCAL_TEST = os.environ.get("PASADO_NB_LOCAL_TEST", "0") == "1"
PIN_TORCH = ["torch==2.11.0", "torchvision==0.26.0"]        # the stack of the October A100 runs
TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
PINS = ["numpy==2.5.1", "scipy==1.18.0", "scikit-learn==1.9.0", "mpmath==1.4.1", "affapy==0.1"]
if LOCAL_TEST:
    REPO, DRIVE_ROOT = os.environ["PASADO_NB_REPO"], os.environ["PASADO_NB_OUT"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    REPO = "/content/Pasado"
    if not os.path.isdir(REPO + "/.git"):
        src = BUNDLE if os.path.exists(BUNDLE) else REPO_URL
        extra = [] if src == BUNDLE else ["--branch", BRANCH]
        subprocess.run(["git", "clone", *extra, src, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", COMMIT], check=True)
    import importlib.metadata as md
    if md.version("torch").split("+")[0] != PIN_TORCH[0].split("==")[1]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIN_TORCH,
                        "--index-url", TORCH_INDEX], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
print("code at", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                                text=True).stdout.strip())

In [ ]:
# 0.2 Environment record and result folder on Drive (one folder per GPU model)
sys.path.insert(0, os.path.join(REPO, "colab"))
import nbutil
GPU = nbutil.gpu_tag()
OUT = os.path.join(DRIVE_ROOT, "01_postfix_rerun", GPU)
CACHE = os.path.join(DRIVE_ROOT, "inductor_cache", GPU)      # compilation is paid once per GPU model
os.makedirs(CACHE, exist_ok=True)
R = nbutil.Runner(REPO, OUT, extra_env={"TORCHINDUCTOR_CACHE_DIR": CACHE})
T = nbutil.LOCAL_TEST
DEV = "cpu" if T else "cuda"
NETS = ["3layer"] if T else ["3layer", "4layer", "5layer", "big"]
if T:
    BIG_CHUNK = 8
else:
    free_mib = int(nbutil.sh("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits").split()[0])
    BIG_CHUNK = 64 if free_mib >= 70 * 1024 else 32
print(json.dumps(R.env_record, indent=1))
print("results ->", OUT, "| big chunk", BIG_CHUNK)
if not T:
    print(nbutil.sh("nvidia-smi"))

In [ ]:
# 0.3 Smoke test (~2 min): the batched path against the sequential loop on this device
ok = R.run("00_smoke_batched_splits", ["experiments/test_batched_splits.py", "--quick", "--device", DEV],
           estimate_min=2)
assert ok, "smoke test failed - stop here and look at the log"

In [ ]:
# A. Split sweep after the fix: bound, time, peak VRAM (~15 min). Mentor's figure.
SPLITS = [1, 2] if T else [1, 2, 4, 8, 16, 32, 64]
IMGS = 1 if T else 3
EST = {"sequential": {"3layer": 1, "4layer": 1.5, "5layer": 2, "big": 5},
       "batched_G": {"3layer": 0.3, "4layer": 0.3, "5layer": 0.3, "big": 3},
       "batched_auto": {"3layer": 0.3, "4layer": 0.3, "5layer": 0.3, "big": 0.7}}
seq_csvs = []
for net in NETS:
    for mode in ("sequential", "batched_G", "batched_auto"):
        out = R.path("A_sweep", f"{net}_{mode}.csv")
        args = ["experiments/measure_split_sweep.py", "--device", DEV, "--dtype", "float64",
                "--networks", net, "--splits", *SPLITS, "--num-images", IMGS, "--eps-indices", 0, 8,
                "--warmup", 1 if T else 2, "--repeats", 1 if T else 5, "--seed", 7, "--out", out,
                "--split-mode", "sequential" if mode == "sequential" else "batched"]
        if mode == "batched_auto":
            args += ["--plane-fit", "auto"]
        if net == "big" and mode != "sequential":
            args += ["--split-chunk", BIG_CHUNK]
        R.run(f"A_{net}_{mode}", args, outputs=[out], estimate_min=EST[mode][net])
        if mode == "sequential":
            seq_csvs.append(out)
R.run("A_soundness", ["experiments/check_sweep_soundness.py", *seq_csvs, "--out", R.path("A_soundness.csv")],
      estimate_min=1)
R.run("A_report", ["experiments/make_checkpoint_report.py", "--postfix", OUT, "--label", "postfix_" + GPU,
                   "--fig-dir", R.path("figures", "x")[:-2]])
try:
    from IPython.display import Image, display
    for f in sorted(os.listdir(os.path.join(OUT, "figures"))):
        display(Image(os.path.join(OUT, "figures", f)))
except ImportError:
    pass

In [ ]:
# B. Share of the CPU part (gelsd and copies) in a pass (~6 min)
for net in NETS:
    out = R.path("B_cpu_share", f"{net}.csv")
    R.run(f"B_cpu_share_{net}", ["experiments/measure_cpu_share.py", "--device", DEV, "--networks", net,
                                 "--splits", *([1, 2] if T else [1, 8, 64]), "--num-images", 1 if T else 3,
                                 "--warmup", 1 if T else 3, "--repeats", 1 if T else 5,
                                 "--big-chunk", BIG_CHUNK, "--out", out],
          outputs=[out], estimate_min={"big": 3}.get(net, 1))

In [ ]:
# C. Equal time on the same instance: the CPU of this VM at n = 1 (~2 min); CUDA side from A
out = R.path("C_cpu_n1.csv")
R.run("C_cpu_n1", ["experiments/measure_split_sweep.py", "--device", "cpu", "--dtype", "float64",
                   "--networks", *NETS, "--splits", 1, "--num-images", 1 if T else 3, "--eps-indices", 0, 8,
                   "--warmup", 1, "--repeats", 1 if T else 3, "--seed", 7, "--out", out],
      outputs=[out], estimate_min=2)

In [ ]:
# D. Device matrix: CUDA against the CPU reference, 30 images x 16 eps, per image (~8 min)
dev_dir = R.path("D_matrix", "cuda", "x")[:-2]
for net in NETS:
    args = ["experiments/postfix_device_matrix.py", "--device", DEV, "--networks", net,
            "--repeat-precise", "--out-dir", dev_dir]
    if T:
        args += ["--num-images", 2, "--eps-indices", 0, 8]
    R.run(f"D_cuda_{net}", args, estimate_min={"big": 4}.get(net, 1))
ref = os.path.join(REPO, "results", "postfix_rerun_2026-11", "cpu_reference")
R.run("D_compare", ["experiments/compare_device_matrix.py", "--cpu", ref, "--device", dev_dir,
                    "--out", R.path("D_summary.csv")] + (["--allow-subset"] if T else []))
# Same-instance check: the Colab CPU against the local reference and against CUDA (subset).
spot = R.path("D_matrix", "colab_cpu", "x")[:-2]
R.run("D_colab_cpu_spot", ["experiments/postfix_device_matrix.py", "--device", "cpu",
                           "--networks", *(["3layer"] if T else ["3layer", "big"]), "--configs", "precise_f64_real",
                           "--num-images", 2 if T else 5, "--eps-indices", 0, 8, 15, "--out-dir", spot],
      estimate_min=1)
R.run("D_spot_vs_reference", ["experiments/compare_device_matrix.py", "--cpu", ref, "--device", spot,
                              "--allow-subset", "--out", R.path("D_spotcheck_summary.csv")])

In [ ]:
# E. Layerwise CPU/CUDA difference, real and complex solver (~5 min)
out = R.path("E_layerwise.csv")
R.run("E_layerwise", ["experiments/measure_layerwise_device_diff.py", "--device", DEV, "--dtype", "float64",
                      "--networks", *NETS, "--num-images", 1 if T else 5,
                      "--eps-indices", *([0] if T else [0, 4, 8, 12, 15]), "--solvers", "real", "complex",
                      "--seed", 7, "--summary-only", "--out", out], outputs=[out], estimate_min=5)

In [ ]:
# F. How far the splits go below one minute per pass (~30-60 min; stop any time, it resumes)
LIMIT = 0.2 if T else 60
for mode in ("batched_auto", "sequential", "batched_G"):       # most useful first
    for net in NETS:
        out = R.path("F_ramp", f"{net}_{mode}.csv")
        R.run(f"F_ramp_{net}_{mode}", ["experiments/measure_split_ramp.py", "--device", DEV, "--networks", net,
                                       "--modes", mode, "--start", 8 if T else 64, "--limit-s", LIMIT,
                                       "--max-splits", 32 if T else 65536, "--out", out],
              outputs=[out], estimate_min=4)
# F2: one full standard run (30 images x 16 eps) per network at the largest n of section A whose
# batched_auto median pass time is at most 125 ms (so 480 passes could fit in a minute).
import csv, statistics
full = []
for net in NETS:
    rows = list(csv.DictReader(open(R.path("A_sweep", f"{net}_batched_auto.csv"))))
    fit = [int(n) for n in sorted({r["n_splits"] for r in rows}, key=int)
           if statistics.mean(float(r["time_median_ms"]) for r in rows if r["n_splits"] == n) <= 125]
    if fit:
        full.append(f"{net}={fit[-1]}")
print("full runs:", full)
if full:
    R.run("F_full_run", ["experiments/measure_split_ramp.py", "--device", DEV, "--full-run", *full,
                         "--big-chunk", BIG_CHUNK, "--out", R.path("F_full_run.csv")], estimate_min=5)

In [ ]:
# Z. Summary tables (re-run after any section). Copy the whole Drive folder OUT into the repo as
# results/postfix_rerun_2026-11/colab_<GPU>/ and run make_checkpoint_report.py there to update CHECKPOINT.md.
subprocess.run([sys.executable, "experiments/make_checkpoint_report.py", "--postfix", OUT,
                "--label", "postfix_" + GPU, "--fig-dir", os.path.join(OUT, "figures")], cwd=REPO)